In [3]:
import numpy as np
import os
import cv2
import time
from landmarkers.mp.hands import MPLiveStreamLandmarker
from landmarkers.inferences import InferenceSequence
from landmarkers.visualization.layers import SequenceLayer, PointsLayer, BBoxLayer
from landmarkers.visualization import Viewer,ViewerBuilder
from landmarkers.visualization.visualizers import LandmarksSequenceVisualizer

2026-01-28 17:53:28.794586: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-28 17:53:28.852743: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-28 17:53:30.331652: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/luis/Documents/projects/python/HandActionDetectionModel/venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


# Preparing Data

## Config

In [4]:
actions = np.array([
	'jump',
	'shoot',
	'none'
])

DATA_PATH = 'HandGestureData'
frame_interval = 1 / 30  # segundos entre frames (~30 FPS)
sequence_length = 15
n_sequences_action = 10

## Creating Folders

In [5]:
for action in actions:
    for seq in range(n_sequences_action):
        os.makedirs(os.path.join(DATA_PATH, action, str(seq)), exist_ok=True)

## Collecting Data

In [1]:

def setup_camera(window_name='Grabación'):
    """Inicializa la cámara y la ventana de visualización."""
    cap = cv2.VideoCapture(0)
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    return cap

def wait_for_start(cap, action, window_name='Grabación'):
    """Muestra la cámara hasta que el usuario pulse 's' para iniciar o 'q' para salir.
       Retorna True si se inició, False si se quiere salir."""
    while True:
        ret, frame = cap.read()
        if not ret:
            continue

        cv2.putText(frame, f'Próximo gesto: {action} — Presiona S para iniciar',
                    (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.imshow(window_name, frame)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('s'):
            return True
        elif key == ord('q'):
            return False

def record_sequence(cap, action, seq, sequence_length, frame_interval, data_path, window_name='Grabación'):
    """Graba una secuencia de frames, detecta manos, extrae características y las guarda."""
    frames_captured = 0
    last_time = time.time()

    while frames_captured < sequence_length:
        current_time = time.time()
        if current_time - last_time >= frame_interval:
            last_time = current_time

            ret, frame = cap.read()
            if not ret:
                continue

            timestamp = int(current_time * 1000)
            result = detect_hands(frame, timestamp)
            frame_display = draw_hand_landmarks(frame, result)

            cv2.putText(frame_display, f'{action} — secuencia {seq+1} frame {frames_captured+1}/{sequence_length}',
                        (10,50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
            cv2.imshow(window_name, frame_display)

            features = extract_hand_features(result)
            save_path = os.path.join(data_path, action, str(seq))
            os.makedirs(save_path, exist_ok=True)
            np.save(os.path.join(save_path, f'{frames_captured}.npy'), features)

            frames_captured += 1

        # Revisar si se quiere salir
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            return False  # Señal para detener todo
    return True  # Secuencia completada

def record_actions(cap, actions, n_sequences_action, sequence_length, frame_interval, data_path):
    """Controla la grabación de todas las acciones."""
    print("INSTRUCCIONES: Presiona 's' para iniciar cada repetición, 'q' para salir.")
    for action in actions:
        print(f"\nPróximo gesto: {action}")

        for seq in range(n_sequences_action):
            print(f"Secuencia {seq+1}/{n_sequences_action}")
            started = wait_for_start(cap, action)
            if not started:
                return  # Se presionó 'q'

            success = record_sequence(cap, action, seq, sequence_length, frame_interval, data_path)
            if not success:
                return  # Se presionó 'q'

            print(f"Secuencia completada: {seq+1}/{n_sequences_action}")
            print("Pulsa 's' para iniciar la siguiente repetición...")

def main():
    cap = setup_camera()
    try:
        record_actions(cap, actions, n_sequences_action, sequence_length, frame_interval, DATA_PATH)
    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Grabación terminada")

if __name__ == "__main__":
    main()


NameError: name 'cv2' is not defined